## Join

1. Создать подключение к базе данных с помощью sqlite3 библиотеки.

In [7]:
import pandas as pd
import sqlite3

conn = sqlite3.connect('../data/checking-logs.sqlite')

2. Создать в базе данных новую таблицу datamart путем объединения таблиц.

In [8]:
query = """
CREATE TABLE IF NOT EXISTS datamart AS
SELECT
    c.uid,
    c.labname,
    c.timestamp AS first_commit_ts,
    MIN(p.datetime) AS first_view_ts
FROM checker AS c
LEFT JOIN pageviews AS p ON c.uid = p.uid
WHERE c.status = 'ready'
AND c.numTrials = 1
AND c.labname IN ("laba04", "laba04s", "laba05", "laba06", "laba06s", "project1")
AND c.uid LIKE "user_%"
GROUP BY c.uid, c.labname
"""

conn.execute("DROP TABLE IF EXISTS datamart")
conn.execute(query)

datamart = pd.read_sql("SELECT * FROM datamart", conn)

datamart.head(10)

,uid,labname,first_commit_ts,first_view_ts
0,user_1,laba04,2020-04-26 17:06:18.462708,2020-04-26 21:53:59.624136
1,user_1,laba04s,2020-04-26 17:12:11.843671,2020-04-26 21:53:59.624136
2,user_1,laba05,2020-05-02 19:15:18.540185,2020-04-26 21:53:59.624136
3,user_1,laba06,2020-05-17 16:26:35.268534,2020-04-26 21:53:59.624136
4,user_1,laba06s,2020-05-20 12:23:37.289724,2020-04-26 21:53:59.624136
5,user_1,project1,2020-05-14 20:56:08.898880,2020-04-26 21:53:59.624136
6,user_10,laba04,2020-04-25 08:24:52.696624,2020-04-18 12:19:50.182714
7,user_10,laba04s,2020-04-25 08:37:54.604222,2020-04-18 12:19:50.182714
8,user_10,laba05,2020-05-01 19:27:26.063245,2020-04-18 12:19:50.182714
9,user_10,laba06,2020-05-19 11:39:28.885637,2020-04-18 12:19:50.182714


3. Используя методы Pandas, создать два датафрейма: test и control.

In [9]:
datamart["first_commit_ts"] = pd.to_datetime(datamart["first_commit_ts"])
datamart["first_view_ts"] = pd.to_datetime(datamart["first_view_ts"])

test = datamart[datamart["first_view_ts"].notna()].copy()
control = datamart[datamart["first_view_ts"].isna()].copy()

control.fillna({"first_view_ts": test["first_view_ts"].mean()}, inplace=True)

test.to_sql("test", conn, if_exists="replace", index=False)
control.to_sql("control", conn, if_exists="replace", index=False)

81

In [10]:
test.head(10)

,uid,labname,first_commit_ts,first_view_ts
0,user_1,laba04,2020-04-26 17:06:18.462708,2020-04-26 21:53:59.624136
1,user_1,laba04s,2020-04-26 17:12:11.843671,2020-04-26 21:53:59.624136
2,user_1,laba05,2020-05-02 19:15:18.540185,2020-04-26 21:53:59.624136
3,user_1,laba06,2020-05-17 16:26:35.268534,2020-04-26 21:53:59.624136
4,user_1,laba06s,2020-05-20 12:23:37.289724,2020-04-26 21:53:59.624136
5,user_1,project1,2020-05-14 20:56:08.898880,2020-04-26 21:53:59.624136
6,user_10,laba04,2020-04-25 08:24:52.696624,2020-04-18 12:19:50.182714
7,user_10,laba04s,2020-04-25 08:37:54.604222,2020-04-18 12:19:50.182714
8,user_10,laba05,2020-05-01 19:27:26.063245,2020-04-18 12:19:50.182714
9,user_10,laba06,2020-05-19 11:39:28.885637,2020-04-18 12:19:50.182714


In [11]:
control.head(10)

,uid,labname,first_commit_ts,first_view_ts
12,user_11,laba05,2020-05-03 21:06:55.970293,2020-04-27 00:40:05.761783552
13,user_11,project1,2020-05-03 23:45:33.673409,2020-04-27 00:40:05.761783552
14,user_12,laba04,2020-04-18 17:07:51.767358,2020-04-27 00:40:05.761783552
15,user_12,laba04s,2020-04-26 15:42:38.070593,2020-04-27 00:40:05.761783552
16,user_12,laba05,2020-05-03 08:39:25.174316,2020-04-27 00:40:05.761783552
17,user_12,laba06,2020-05-19 08:41:51.082304,2020-04-27 00:40:05.761783552
18,user_12,project1,2020-05-14 18:17:40.702024,2020-04-27 00:40:05.761783552
19,user_13,laba04,2020-04-25 17:56:16.919886,2020-04-27 00:40:05.761783552
20,user_13,laba04s,2020-04-26 22:21:04.093297,2020-04-27 00:40:05.761783552
21,user_13,laba05,2020-05-02 19:31:35.398386,2020-04-27 00:40:05.761783552


4. Закрыть соединение

In [12]:
conn.close()